## StatsBomb Open Data Audit

This notebook inspects the available competitions, seasons, matches, and teams
before constructing a multi-league expected threat model.
# Build a General xT Training Dataset

This notebook audits the local StatsBomb Open Data repository and constructs
a match-level training dataset for a general expected threat model.

The training dataset focuses on senior men's domestic league matches with
sufficient competition-season coverage.

In [7]:
from pathlib import Path
import json

import pandas as pd
import numpy as np
DATA_ROOT = Path(
    r"D:\UIUC\Projects\World Cup 2026\archive\data"
)

COMPETITIONS_PATH = DATA_ROOT / "competitions.json"
MATCHES_ROOT = DATA_ROOT / "matches"
EVENTS_ROOT = DATA_ROOT / "events"


In [8]:
def load_json(path: Path):
    """
    Load a JSON file and return its Python object.
    """
    #This is a docstring 
    with path.open("r", encoding="utf-8") as file: #"with" is used to create a context manager, auto close the file when use is done
        return json.load(file)#read json as python dictionary

In [9]:
competitions = pd.DataFrame(
    load_json(COMPETITIONS_PATH)
)

display(
    competitions[["country_name",
        "competition_name",
        "season_name",
        "competition_id",
        "season_id",
        "competition_gender",
        "competition_youth",
        "competition_international",]]
    .sort_values(
        ["country_name", "competition_name", "season_name"]
    )
    .reset_index(drop=True)
)

,country_name,competition_name,season_name,competition_id,season_id,competition_gender,competition_youth,competition_international
0,Africa,African Cup of Nations,2023,1267,107,male,False,True
1,Argentina,Liga Profesional,1981,81,275,male,False,False
2,Argentina,Liga Profesional,1997/1998,81,48,male,False,False
3,England,FA Women's Super League,2018/2019,37,4,female,False,False
4,England,FA Women's Super League,2019/2020,37,42,female,False,False
...,...,...,...,...,...,...,...,...
75,Spain,La Liga,2020/2021,11,90,male,False,False
76,Spain,Liga F,2023/2024,182,281,female,False,False
77,United States of America,Major League Soccer,2023,44,107,male,False,False
78,United States of America,NWSL,2018,49,3,female,False,False


In [10]:
selected_competitions = competitions[
    (competitions["competition_gender"] == "male")
    & (competitions["competition_youth"] == False)
    & (competitions["competition_international"] == False)
].copy()

display(
    selected_competitions[
        [
            "country_name",
            "competition_name",
            "season_name",
            "competition_id",
            "season_id",
        ]
    ].head()
)


,country_name,competition_name,season_name,competition_id,season_id
0,Germany,1. Bundesliga,2023/2024,9,281
1,Germany,1. Bundesliga,2015/2016,9,27
3,Europe,Champions League,2018/2019,16,4
4,Europe,Champions League,2017/2018,16,1
5,Europe,Champions League,2016/2017,16,2


In [11]:
all_matches = []
for row in selected_competitions.itertuples(index=False):#itertuples 按行遍历dataframe，把每一行转换为元组对象，False不携带行索引.

    match_path = (
        MATCHES_ROOT
        / str(row.competition_id)
        / f"{row.season_id}.json"
    )

    if not match_path.exists():#判断json match 文件是否存在
        continue
#使用normalize扁平化嵌套json，
    matches = pd.json_normalize(
        load_json(match_path)
    )

    matches["competition_name"] = row.competition_name
    matches["season_name"] = row.season_name
    matches["country_name"] = row.country_name

    all_matches.append(matches)

training_matches = pd.concat(
    all_matches,
    ignore_index=True,
)

In [12]:
training_matches["event_path"] = (
    training_matches["match_id"]
    .astype(int)
    .apply(
        lambda match_id:
        EVENTS_ROOT / f"{match_id}.json"
    )
)
training_matches = training_matches[
    [
        "match_id",
        "match_date",
        "country_name",
        "competition_name",
        "season_name",
        "home_team.home_team_name",
        "away_team.away_team_name",
        "event_path",
    ]
].copy()

In [13]:
print("Total matches:", len(training_matches))

print(
    "Total competitions:",
    training_matches["competition_name"].nunique(),
)

print(
    "Total seasons:",
    training_matches[
        ["competition_name", "season_name"]
    ].drop_duplicates().shape[0],
)

display(training_matches.head())

Total matches: 2317
Total competitions: 12
Total seasons: 54


,match_id,match_date,country_name,competition_name,season_name,home_team.home_team_name,away_team.away_team_name,event_path
0,3895292,2024-04-06,Germany,1. Bundesliga,2023/2024,Union Berlin,Bayer Leverkusen,D:\UIUC\Projects\World Cup 2026\archive\data\e...
1,3895320,2024-04-27,Germany,1. Bundesliga,2023/2024,Bayer Leverkusen,VfB Stuttgart,D:\UIUC\Projects\World Cup 2026\archive\data\e...
2,3895158,2023-12-03,Germany,1. Bundesliga,2023/2024,Bayer Leverkusen,Borussia Dortmund,D:\UIUC\Projects\World Cup 2026\archive\data\e...
3,3895107,2023-10-08,Germany,1. Bundesliga,2023/2024,Bayer Leverkusen,FC Köln,D:\UIUC\Projects\World Cup 2026\archive\data\e...
4,3895340,2024-05-12,Germany,1. Bundesliga,2023/2024,Bochum,Bayer Leverkusen,D:\UIUC\Projects\World Cup 2026\archive\data\e...


# read every event json and make a whole dataFrame for every pass

First, observe the event JSON of a game

In [14]:
sample_match = training_matches.iloc[0]

sample_match_id = sample_match["match_id"]
sample_event_path = sample_match["event_path"]

events = load_json(sample_event_path)

print("Match ID:", sample_match_id)
print("Number of events:", len(events))
print("Data type:", type(events))

Match ID: 3895292
Number of events: 3843
Data type: <class 'list'>


In [15]:
first_pass = next( #next indicates extracting the first event from those that meet the conditions.
    event
    for event in events
    if event["type"]["name"] == "Pass"
)
first_pass

{'id': '6c546c19-5f61-4236-a1ad-0b6f5cded692',
 'index': 5,
 'period': 1,
 'timestamp': '00:00:01.010',
 'minute': 0,
 'second': 1,
 'type': {'id': 30, 'name': 'Pass'},
 'possession': 2,
 'possession_team': {'id': 190, 'name': 'Union Berlin'},
 'play_pattern': {'id': 9, 'name': 'From Kick Off'},
 'team': {'id': 190, 'name': 'Union Berlin'},
 'player': {'id': 24243, 'name': 'Brenden Aaronson'},
 'position': {'id': 19, 'name': 'Center Attacking Midfield'},
 'location': [61.0, 40.1],
 'duration': 0.697031,
 'related_events': ['7ee5b148-1a76-42ec-8744-332edb0caad6'],
 'pass': {'recipient': {'id': 42822, 'name': 'András Schäfer'},
  'length': 2.7730849,
  'angle': -2.6940727,
  'height': {'id': 1, 'name': 'Ground Pass'},
  'end_location': [58.5, 38.9],
  'body_part': {'id': 40, 'name': 'Right Foot'},
  'type': {'id': 65, 'name': 'Kick Off'}}}

In [16]:
def extract_actions(event_path, match_id):
    events = load_json(event_path)
    action_rows = []
    for event in events:
        event_type = event.get("type",{}).get("name")
        if event_type not in ["Pass", "Carry", "Shot"]:
            continue
        location = event.get("location")
        if location is None:
            continue
        team_data = event.get("team")
        player_data = event.get("player")
        action = {
            "match_id": int(match_id),
            "event_id": event.get("id"),
            "period": event.get("period"),
            "minute": event.get("minute"),
            "second": event.get("second"),
            "team": (
                team_data.get("name")
                if team_data is not None
                else None
            ),
            "player": (
                player_data.get("name")
                if player_data is not None
                else None
            ),
            "event_type": event_type,
            "start_x": location[0],
            "start_y": location[1],
            "end_x": None,
            "end_y": None,
            "success": None,
            "goal": False,
        }
        if event_type == "Pass":
            pass_data = event.get("pass", {})
            end_location = pass_data.get("end_location")
            pass_outcome = pass_data.get("outcome")

            if end_location is not None:
                action["end_x"]=end_location[0]
                action["end_y"]= end_location[1]
            action["success"] = pass_outcome is None

        elif event_type == "Carry":# carry must success because it has location and end loction already.
            carry_data = event.get("carry", {})
            end_location = carry_data.get("end_location")

            if end_location is not None:
                action["end_x"] = end_location[0]
                action["end_y"] = end_location[1]

            action["success"] = True

        elif event_type == "Shot":

            shot_data = event.get("shot", {})
            outcome = shot_data.get("outcome", {})
            outcome_name = outcome.get("name")

            action["goal"] = outcome_name == "Goal"

        action_rows.append(action)
    return pd.DataFrame(action_rows)


In [17]:
actions = extract_actions(sample_event_path, sample_match_id)

print(actions.columns)

Index(['match_id', 'event_id', 'period', 'minute', 'second', 'team', 'player',
       'event_type', 'start_x', 'start_y', 'end_x', 'end_y', 'success',
       'goal'],
      dtype='str')


I didn't extract dribble because the value of it are only complete and incomplete. I don't have the location position.

In [18]:
sample_actions = extract_actions(sample_event_path,sample_match_id)
display(sample_actions.head(10))
sample_actions["event_type"].value_counts()

,match_id,event_id,period,minute,second,team,player,event_type,start_x,start_y,end_x,end_y,success,goal
0,3895292,6c546c19-5f61-4236-a1ad-0b6f5cded692,1,0,1,Union Berlin,Brenden Aaronson,Pass,61.0,40.1,58.5,38.9,True,False
1,3895292,a7fdbea2-aabe-498d-ac0d-ebd6e3edabca,1,0,1,Union Berlin,András Schäfer,Pass,58.8,38.6,35.9,50.9,True,False
2,3895292,00724eb9-1285-4be4-8cc8-9231ccf63110,1,0,3,Union Berlin,Danilho Doekhi,Carry,35.9,50.9,37.9,52.2,True,False
3,3895292,f3647dd4-0731-472b-b2e8-31b01e772fc7,1,0,4,Union Berlin,Danilho Doekhi,Pass,37.9,52.2,86.1,6.0,True,False
4,3895292,ba8350af-91ff-4a29-a646-d88829863907,1,0,8,Union Berlin,Robin Gosens,Pass,84.3,6.2,94.2,16.7,False,False
5,3895292,7b049f8b-d62e-412b-9d79-471b00f9e8b4,1,0,9,Bayer Leverkusen,Odilon Kossonou,Pass,25.9,63.4,36.1,63.4,False,False
6,3895292,f53c7a00-a07a-4776-bbce-bff7a392d4dc,1,0,10,Union Berlin,András Schäfer,Carry,84.0,16.7,83.9,15.6,True,False
7,3895292,a829fba7-f2bf-4151-ae58-2804ae73814b,1,0,11,Bayer Leverkusen,Robert Andrich,Carry,33.2,69.2,33.2,70.9,True,False
8,3895292,15133aa4-99d5-49bb-997a-07b497aa3fa0,1,0,12,Bayer Leverkusen,Robert Andrich,Pass,33.2,70.9,35.8,70.9,False,False
9,3895292,9493839c-5c1d-4ee6-b9d7-b7cb665bcb40,1,0,24,Bayer Leverkusen,Nathan Tella,Pass,36.1,80.0,45.9,73.9,True,False


event_type
Pass     1080
Carry     905
Shot       29
Name: count, dtype: int64

In [19]:
#check pass
display(
    sample_actions[
        sample_actions["event_type"] == "Pass"
    ].head(5)
)

,match_id,event_id,period,minute,second,team,player,event_type,start_x,start_y,end_x,end_y,success,goal
0,3895292,6c546c19-5f61-4236-a1ad-0b6f5cded692,1,0,1,Union Berlin,Brenden Aaronson,Pass,61.0,40.1,58.5,38.9,True,False
1,3895292,a7fdbea2-aabe-498d-ac0d-ebd6e3edabca,1,0,1,Union Berlin,András Schäfer,Pass,58.8,38.6,35.9,50.9,True,False
3,3895292,f3647dd4-0731-472b-b2e8-31b01e772fc7,1,0,4,Union Berlin,Danilho Doekhi,Pass,37.9,52.2,86.1,6.0,True,False
4,3895292,ba8350af-91ff-4a29-a646-d88829863907,1,0,8,Union Berlin,Robin Gosens,Pass,84.3,6.2,94.2,16.7,False,False
5,3895292,7b049f8b-d62e-412b-9d79-471b00f9e8b4,1,0,9,Bayer Leverkusen,Odilon Kossonou,Pass,25.9,63.4,36.1,63.4,False,False


# First dispose 20 matches

In [20]:
matches_to_process = training_matches.head(20)

action_tables = []

for row in matches_to_process.itertuples(index=False):

    match_actions = extract_actions(
        event_path=row.event_path,
        match_id=row.match_id,
    )

    action_tables.append(match_actions)

actions = pd.concat(
    action_tables,
    ignore_index=True,
)
print("Matches processed:", len(matches_to_process))
print("Actions extracted:", len(actions))

Matches processed: 20
Actions extracted: 42245


Check if pass and carry have a terminus. Check the pass success rate to determine if the data is correctly extracted.

In [21]:
move_actions = actions[
    actions["event_type"].isin(["Pass", "Carry"])
]

print(
    move_actions[
        ["end_x", "end_y"]
    ].isna().sum()
)

end_x    0
end_y    0
dtype: int64


In [22]:
passes = actions[
    actions["event_type"] == "Pass"
]

print(
    passes["success"].value_counts(
        normalize=True
    )
)

success
True     0.85425
False    0.14575
Name: proportion, dtype: float64


In [23]:
matches_to_process = training_matches
action_tables = []

for number, row in enumerate(
    matches_to_process.itertuples(index=False),
    start=1,
):

    match_actions = extract_actions(
        event_path=row.event_path,
        match_id=row.match_id,
    )

    action_tables.append(match_actions)

    if number % 100 == 0:
        print(
            f"Processed {number} "
            f"of {len(matches_to_process)} matches"
        )

actions = pd.concat(
    action_tables,
    ignore_index=True,
)

Processed 100 of 2317 matches
Processed 200 of 2317 matches
Processed 300 of 2317 matches
Processed 400 of 2317 matches
Processed 500 of 2317 matches
Processed 600 of 2317 matches
Processed 700 of 2317 matches
Processed 800 of 2317 matches
Processed 900 of 2317 matches


KeyboardInterrupt: 

In [ ]:
len(actions)

4133519

However, before saving, it is recommended to combine the event and season information into the "actions". Otherwise, when filtering for "Barcelona matches of a certain season", the "actions" only have the match_id and the team name, and it is unknown which event and season they belong to. 
Merge the competition information into actions

In [24]:
match_information = training_matches[
    [
        "match_id",
        "match_date",
        "country_name",
        "competition_name",
        "season_name",
        "home_team.home_team_name",
        "away_team.away_team_name",
    ]
].drop_duplicates(subset="match_id")


In [25]:


OUTPUT_ROOT = Path(r"D:\UIUC\Projects\World Cup 2026\Learning\output")
processed_dir = OUTPUT_ROOT / "processed_data"
processed_dir.mkdir(exist_ok=True, parents=True)

actions.to_parquet(
    processed_dir / "actions.parquet",
    index=False,
)
# 保存 match_information 表
match_information.to_parquet(
    processed_dir / "match_metadata.parquet",
    index=False,
)